# Generate synthetic bakery data
Run top to bottom. You get a zip of CSVs for the IEOR 145 bakery project.

## 1. Setup
Works locally and in Colab. In Colab it clones the repo first.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path("ieor145").exists():
        subprocess.run(["git", "clone", "https://github.com/zhouhaitz/ieor145"], check=True)
    os.chdir("ieor145")
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

sys.path.insert(0, str(Path.cwd()))
print("Repo root:", Path.cwd())

## 2. Pick a scenario
The baseline is a neighborhood bakery open 7am to 5pm. Change a knob below to try another world.

In [ ]:
from bakery_sim.config import load_scenario

scenario = load_scenario("configs/baseline.json")

In [ ]:
# Uncomment to change a knob (each line returns a new scenario):
# scenario = scenario.with_overrides(supply_ratio=1.0)          # bake less (baked / expected demand)
# scenario = scenario.with_overrides(demand_multiplier_cv=0.3)  # more day-to-day swings
# scenario = scenario.with_overrides(salvage_fraction=0.0)      # leftovers are thrown away
# scenario = scenario.with_overrides(carryover_enabled=False)   # sourdough is not sold day-old

## 3. Generate
Writes two datasets to `data/`: sourdough sold day-old (`_carryover_on`) and not (`_carryover_off`).
Use `door_counter/` for modeling (customer type hidden). `truth/` adds hidden columns for checking; `register/` is purchases only.

In [ ]:
from bakery_sim.generate import generate_carryover_pair

results = generate_carryover_pair(scenario, "data")
list(results)

## 4. Report card
Each row checks one number against a sourced band (see `docs/sources.md`). A failing row means revisit the inputs, never edit the outputs.
`realized_hourly_shape_error` has no band: it includes day-to-day noise. The graded `hourly_shape_error` is the model's expected traffic vs Popular Times.

In [ ]:
from bakery_sim.report_card import format_report_card

for name, (_, card) in results.items():
    print(f"{scenario.scenario_id}_{name}")
    print(format_report_card(card), end="\n\n")

## 5. Quick look
Three sanity plots for the carryover-on dataset.

In [ ]:
import matplotlib.pyplot as plt

dataset, _ = results["carryover_on"]
days = dataset.days[["day_index", "day_of_week"]]
visits = dataset.visits().merge(days, on="day_index")

Simulated customers per hour vs real Google Popular Times (weekday and Saturday).

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for axis, day_of_week in zip(axes, ["Wed", "Sat"], strict=True):
    counts = visits[visits["day_of_week"] == day_of_week].groupby("hour").size()
    simulated_share = counts.reindex(scenario.open_hours, fill_value=0) / counts.sum()
    axis.bar(scenario.open_hours, simulated_share, color="#9ecae1", label="Simulated")
    axis.plot(scenario.open_hours, scenario.observed_hourly_shape[day_of_week], "o-", color="#08519c", label="Popular Times")
    axis.set_title(day_of_week)
    axis.set_xlabel("Hour of day (24h clock)")
axes[0].set_ylabel("Share of the day's customers (fraction)")
axes[0].legend()
plt.tight_layout()
plt.show()

Units baked, sold and wasted per day, by product.

In [ ]:
units_per_day = dataset.inventory.groupby("product_id")[["baked_units", "sold_units", "wasted_units"]].sum() / len(dataset.days)
units_per_day = units_per_day.rename(columns={"baked_units": "Baked", "sold_units": "Sold", "wasted_units": "Wasted"})
units_per_day.plot.bar(figsize=(7, 3.5), color=["#9ecae1", "#3182bd", "#e6550d"], rot=0)
plt.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.xlabel("Product")
plt.ylabel("Units per day")
plt.tight_layout()
plt.show()

How often each price level was on the shelf in the history log.

In [ ]:
level_share = dataset.prices["price_level"].value_counts(normalize=True).sort_index() * 100
level_share.plot.bar(figsize=(6, 3.5), color="#3182bd", rot=0)
plt.xlabel("Price level (fraction of menu price)")
plt.ylabel("Share of shelf-hours (%)")
plt.tight_layout()
plt.show()

## 6. Download
In Colab this downloads the zips. Locally it prints where they are.

In [ ]:
zip_paths = [Path("data") / f"{scenario.scenario_id}_{name}.zip" for name in results]
if IN_COLAB:
    from google.colab import files

    for zip_path in zip_paths:
        files.download(str(zip_path))
else:
    for zip_path in zip_paths:
        print(zip_path.resolve())